In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = [
    "How do I reset my password if I forget it?",
    "I cannot remember my login credentials and need to recover my account.",
    "What is the monthly subscription price for the pro plan?",
    "How much does the premium tier cost per month?",
]
embeddings = model.encode(sentences)
print(embeddings.shape)   # (4, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(4, 384)


In [ ]:
faq_texts = [
    "How do I reset my password if I forgot it",
    "I cannot remember my login credentials and need to recover my account",
    "What is the monthly subscription price for the Pro plan",
    "How much does the premium tier cost per month",
]

faq_embeddings = model.encode(faq_texts)
print(faq_embeddings.shape)
print(faq_embeddings[0][:5])

(4, 384)
[-0.00544345 -0.04238848 -0.07683909  0.01654602 -0.02997587]


In [ ]:
import numpy as np

def cosine_similarity(u, v):
    u, v = np.array(u), np.array(v)
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

password_reset = cosine_similarity(faq_embeddings[0], faq_embeddings[1])
plans = cosine_similarity(faq_embeddings[2], faq_embeddings[3])
print("Similarity between sentences[0],[1]:", password_reset)
print("Similarity between sentences[2],[3]:", plans)

Similarity between sentences[0],[1]: 0.7119190096855164
Similarity between sentences[2],[3]: 0.5751464366912842


In [ ]:
import numpy as np
import re

In [ ]:
stop = {"a", "an", "the", "on", "in", "of", "is", "are", "to"}

In [ ]:
def tokenize(s):
    toks = re.findall(r"[a-z]+", s.lower())
    return [t for t in toks if t not in stop]

In [ ]:
docs = {
    "d1": "The cat sat on the mat",
    "d2": "A feline rested on the rug",
    "d3": "The stock market crashed today",
}

In [ ]:
vocab = sorted(set(w for d in docs.values() for w in tokenize(d)))
vocab

['cat',
 'crashed',
 'feline',
 'market',
 'mat',
 'rested',
 'rug',
 'sat',
 'stock',
 'today']

In [ ]:
print("Vocabulary (stopwords removed):",vocab)

Vocabulary (stopwords removed): ['cat', 'crashed', 'feline', 'market', 'mat', 'rested', 'rug', 'sat', 'stock', 'today']


In [ ]:
def vectorize(s):
  toks = tokenize(s)
  return np.array([toks.count(w) for w in vocab], dtype=float)

In [ ]:
vecs = {k: vectorize(v) for k, v in docs.items()}
print(vecs)

{'d1': array([1., 0., 0., 0., 1., 0., 0., 1., 0., 0.]), 'd2': array([0., 0., 1., 0., 0., 1., 1., 0., 0., 0.]), 'd3': array([0., 1., 0., 1., 0., 0., 0., 0., 1., 1.])}


In [ ]:
def cosine_similarity(u, v):
  if np.linalg.norm(u)==0 or np.linalg.norm(v)==0:
    return 0.0
  return float(np.dot(u, v)/(np.linalg.norm(u)*np.linalg.norm(v)))


In [ ]:
print("Keyword cosine similarity, d1 vs d2 (cat/mat vs feline/rug):", cosine_similarity(vecs["d1"], vecs["d2"]))
print("Keyword cosine similarity, d1 vs d3 (cat/mat vs stock market):", cosine_similarity(vecs["d1"], vecs["d3"]))

Keyword cosine similarity, d1 vs d2 (cat/mat vs feline/rug): 0.0
Keyword cosine similarity, d1 vs d3 (cat/mat vs stock market): 0.0


In [ ]:
import csv

with open("/content/pJLYme3LfqUZaICJ.csv") as f:
    reader = csv.DictReader(f)
    faqs = {int(row["id"]): row["text"] for row in reader}

for i, text in faqs.items():
    print(i, "-", text)

1 - How do I reset my password if I forgot it
2 - I cannot remember my login credentials and need to recover my account
3 - What is the monthly subscription price for the Pro plan
4 - How much does the premium tier cost per month
5 - Can I export my notes to PDF format
6 - Is there a way to download my notes as a PDF file
7 - Does the app work on both Android and iPhone
8 - My notes are not syncing between my laptop and my phone
9 - How do I delete my account permanently
10 - Where can I change my email notification preferences


In [21]:
vocab2 = sorted(set(w for t in faqs.values() for w in tokenize(t)))
def vectorize2(s):
    toks = tokenize(s)
    return np.array([toks.count(w) for w in vocab2], dtype=float)

fvecs = {i: vectorize2(t) for i, t in faqs.items()}

pairs = [(1, 2), (3, 4), (5, 6),(7,8),(9,10)]
for a, b in pairs:
    print(f"Keyword similarity ({a} vs {b}):", round(cosine_similarity(fvecs[a], fvecs[b]), 4))
    print(" ", faqs[a], "||", faqs[b])

Keyword similarity (1 vs 2): 0.3203
  How do I reset my password if I forgot it || I cannot remember my login credentials and need to recover my account
Keyword similarity (3 vs 4): 0.0
  What is the monthly subscription price for the Pro plan || How much does the premium tier cost per month
Keyword similarity (5 vs 6): 0.4009
  Can I export my notes to PDF format || Is there a way to download my notes as a PDF file
Keyword similarity (7 vs 8): 0.0945
  Does the app work on both Android and iPhone || My notes are not syncing between my laptop and my phone
Keyword similarity (9 vs 10): 0.2673
  How do I delete my account permanently || Where can I change my email notification preferences


In [22]:
faq_embeddings = model.encode(list(faqs.values()))
print(faq_embeddings.shape)
print(faq_embeddings[0][:5])

(10, 384)
[-0.00544345 -0.04238848 -0.07683909  0.01654602 -0.02997587]


In [23]:
pair1 = cosine_similarity(faq_embeddings[0], faq_embeddings[1])
pair2 = cosine_similarity(faq_embeddings[2], faq_embeddings[3])
pair3 = cosine_similarity(faq_embeddings[4], faq_embeddings[5])
pair4 = cosine_similarity(faq_embeddings[6], faq_embeddings[7])
pair5 = cosine_similarity(faq_embeddings[8], faq_embeddings[9])
print("Similarity for first pair:", pair1)
print("Similarity for second pair:", pair2)
print("Similarity for third pair:", pair3)
print("Similarity for fourth pair:", pair4)
print("Similarity for fifth pair:", pair5)

Similarity for first pair: 0.7119190096855164
Similarity for second pair: 0.5751464366912842
Similarity for third pair: 0.8743846416473389
Similarity for fourth pair: 0.2905324697494507
Similarity for fifth pair: 0.1717047095298767


In [26]:
import csv
import re
import numpy as np

STOP = {"a", "the", "on", "in", "s", "of", "is", "to", "my", "i", "do", "does", "can", "how", "what", "where"}

def tokenize(s):
    toks = re.findall(r"[a-z]+", s.lower())
    return [t for t in toks if t not in STOP]

with open("/content/pJLYme3LfqUZaICJ.csv") as f:
    reader = csv.DictReader(f)
    faqs = {int(row["id"]): row["text"] for row in reader}

vocab = sorted(set(w for t in faqs.values() for w in tokenize(t)))

def vectorize(s):
    toks = tokenize(s)
    return np.array([toks.count(w) for w in vocab], dtype=float)

faq_vectors = {i: vectorize(t) for i, t in faqs.items()}

def cosine_similarity(u, v):
    if (np.linalg.norm(u) == 0 or np.linalg.norm(v) == 0):
        return 0.0
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

def keyword_search(query, top_k=3):
    qvec = ___
    scores = [(i, cosine_similarity(___, ___)) for i in faqs]
    scores.sort(key=lambda x: -x[1])
    return scores[:___]

print(keyword_search("I want to get a refund of my subscription"))
print(keyword_search("download my notes as PDF"))


ValueError: could not convert string to float: np.str_('')